In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
BASE_DIR = Path('/mimer/NOBACKUP/groups/naiss2024-6-297/cache/bertopic_bootstrapped')
MIN_TOPIC_SIZES = [5, 10, 15, 20, 25]

# Load all pilot discovery summaries across min_topic_size values
all_rows = []
for mts in MIN_TOPIC_SIZES:
    pilot_dir = BASE_DIR / f'pilot_discovery_{mts}'
    files = sorted(pilot_dir.glob('discovery_summary_bootstrap_*.json'))
    print(f'min_topic_size={mts:>3d}: {len(files):>2d}/10 bootstraps in {pilot_dir.name}/')
    for path in files:
        with open(path, 'r') as f:
            rec = json.load(f)
        rec['min_topic_size'] = mts
        all_rows.append(rec)

df_all = pd.DataFrame(all_rows)
df_all['outlier_rate'] = df_all['outlier_sentence_count'] / df_all['n_sentences']
print(f'\nTotal records loaded: {len(df_all)}')
df_all.head()

In [ ]:
# For each min_topic_size: mean ± std of key metrics across bootstraps

metrics = ['discovered_topics_excl_outlier', 'outlier_sentence_count', 'outlier_rate']

summary = (
    df_all.groupby('min_topic_size')[metrics]
    .agg(['mean', 'std', 'min', 'max'])
)
# Flatten multi-level columns for readability
summary.columns = [f'{m}_{stat}' for m, stat in summary.columns]
summary = summary.round(3)
summary

In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# num of topics without outliers
ax = axes[0]
grouped = df_all.groupby('min_topic_size')['discovered_topics_excl_outlier']
means = grouped.mean()
stds = grouped.std()
ax.errorbar(means.index, means.values, yerr=stds.values, fmt='o-', capsize=4, color='tab:blue')
ax.set_xlabel('min_topic_size (= min_cluster_size)')
ax.set_ylabel('Discovered topics (excl outlier)')
ax.set_title('Topics discovered')
ax.set_xticks(MIN_TOPIC_SIZES)

# outlier rate
ax = axes[1]
grouped = df_all.groupby('min_topic_size')['outlier_rate']
means = grouped.mean()
stds = grouped.std()
ax.errorbar(means.index, means.values, yerr=stds.values, fmt='s-', capsize=4, color='tab:orange')
ax.set_xlabel('min_topic_size (= min_cluster_size)')
ax.set_ylabel('Outlier rate (fraction of sentences)')
ax.set_title('Outlier rate')
ax.set_xticks(MIN_TOPIC_SIZES)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

# outlier count
ax = axes[2]
grouped = df_all.groupby('min_topic_size')['outlier_sentence_count']
means = grouped.mean()
stds = grouped.std()
ax.errorbar(means.index, means.values, yerr=stds.values, fmt='^-', capsize=4, color='tab:red')
ax.set_xlabel('min_topic_size (= min_cluster_size)')
ax.set_ylabel('Outlier sentences (topic = -1)')
ax.set_title('Outlier sentence count')
ax.set_xticks(MIN_TOPIC_SIZES)

fig.suptitle('Pilot Discovery: Effect of min_topic_size across bootstraps', fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# Change these to explore different settings
MTS_TO_INSPECT = 10
BOOTSTRAP_TO_INSPECT = 0

pilot_dir = BASE_DIR / f'pilot_discovery_{MTS_TO_INSPECT}'
topic_info_path = pilot_dir / f'topic_info_bootstrap_{BOOTSTRAP_TO_INSPECT:02d}.csv'

if not topic_info_path.exists():
    print(f'File not found: {topic_info_path}')
    print('Jobs may still be running. Check with: squeue -u $USER')
else:
    topic_info = pd.read_csv(topic_info_path)
    print(f'min_topic_size={MTS_TO_INSPECT}, bootstrap={BOOTSTRAP_TO_INSPECT}')
    print(f'Columns: {list(topic_info.columns)}')
    print(f'Total topics (incl outlier): {len(topic_info)}')

    # Top topics by size (excluding outlier topic -1)
    topics_main = topic_info[topic_info['Topic'] != -1].sort_values('Count', ascending=False)
    print(f'Topics (excl outlier): {len(topics_main)}')
    topics_main.head(20)

In [ ]:
# top words per topic

def parse_top_words(row, max_words=10):
    """Extract readable top words from BERTopic topic_info row."""
    if 'Representation' in row and pd.notna(row['Representation']):
        txt = str(row['Representation']).strip('[]')
        words = [w.strip().strip("'").strip('"') for w in txt.split(',') if w.strip()]
        return ', '.join(words[:max_words])
    if 'Name' in row and pd.notna(row['Name']):
        name = str(row['Name'])
        if '_' in name:
            return ', '.join(name.split('_')[1:1+max_words])
        return name
    return ''

if topic_info_path.exists():
    inspect_cols = ['Topic', 'Count']
    if 'Name' in topic_info.columns:
        inspect_cols.append('Name')

    topics_words = topics_main.copy()
    topics_words['TopWords'] = topics_words.apply(parse_top_words, axis=1)

    print(f'Top 30 topics by size (min_topic_size={MTS_TO_INSPECT}, bootstrap={BOOTSTRAP_TO_INSPECT}):')
    with pd.option_context('display.max_colwidth', None):
        display(topics_words[inspect_cols + ['TopWords']].head(30))

In [ ]:
# outlier sentences

if topic_info_path.exists():
    outlier_row = topic_info[topic_info['Topic'] == -1]

    if outlier_row.empty:
        print('No topic -1 found.')
    else:
        display(outlier_row)

        if 'Count' in outlier_row.columns:
            outlier_count = int(outlier_row['Count'].iloc[0])
            total_chunks = int(topic_info['Count'].sum())
            print(f'Outlier: {outlier_count} / {total_chunks} sentences ({outlier_count/total_chunks:.2%})')

        rep_col = None
        for c in ['Representative_Docs', 'Representative Docs']:
            if c in outlier_row.columns:
                rep_col = c
                break
        if rep_col is not None:
            print(f'\nRepresentative outlier content:')
            print(outlier_row[rep_col].iloc[0])
        else:
            print('\nNo representative-doc column in topic_info.')

In [ ]:
# topic size distribution per setting

fig, axes = plt.subplots(1, len(MIN_TOPIC_SIZES), figsize=(4 * len(MIN_TOPIC_SIZES), 4), sharey=True)
bootstrap_to_compare = 0

for i, mts in enumerate(MIN_TOPIC_SIZES):
    ax = axes[i]
    ti_path = BASE_DIR / f'pilot_discovery_{mts}' / f'topic_info_bootstrap_{bootstrap_to_compare:02d}.csv'
    if not ti_path.exists():
        ax.set_title(f'mts={mts}\n(not ready)')
        continue
    ti = pd.read_csv(ti_path)
    sizes = ti.loc[ti['Topic'] != -1, 'Count'].values
    ax.hist(sizes, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_title(f'mts={mts}\n{len(sizes)} topics')
    ax.set_xlabel('Topic size (sentences)')
    if i == 0:
        ax.set_ylabel('Number of topics')
    ax.axvline(x=mts, color='red', linestyle='--', alpha=0.7, label=f'min={mts}')
    ax.legend(fontsize=8)

fig.suptitle(f'Topic size distributions (bootstrap {bootstrap_to_compare})', fontsize=13, y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# decision summary

print(f'{"mts":>5s}  {"topics_mean":>12s}  {"topics_std":>10s}  {"outlier%_mean":>13s}  {"nr_topics upper":>16s}')

for mts in MIN_TOPIC_SIZES:
    subset = df_all[df_all['min_topic_size'] == mts]
    if subset.empty:
        print(f'{mts:>5d}  {"(no data)":>12s}')
        continue
    t_mean = subset['discovered_topics_excl_outlier'].mean()
    t_std = subset['discovered_topics_excl_outlier'].std()
    o_mean = subset['outlier_rate'].mean()
    # upper bound: mean - 1*std, stay below discovered count
    upper = int(np.floor(t_mean - t_std)) if not np.isnan(t_std) else int(t_mean)
    print(f'{mts:>5d}  {t_mean:>12.1f}  {t_std:>10.1f}  {o_mean:>12.1%}  {upper:>16d}')

print()
print('nr_topics must be < discovered topics (BERTopic skips reduction otherwise)')
